# Sinhala-Tamil Pivot MT — Proof of Concept

**Research:** Using English as a pivot language to overcome the Sinhala-Tamil data scarcity barrier.

**Hypothesis:** A cascade Si→En→Ta pipeline using pretrained models will substantially
outperform a direct Transformer trained on only ~25k sentence pairs.

---

## Pipeline Overview

```
data/raw/corpus.{si,ta}
        │
        ▼
Phase 1 — preprocess.py       (clean, normalise, length-filter, split)
        │
        ▼
Phase 1b — labse_filter.py    (LaBSE cosine similarity ≥ 0.85)
        │
        ▼
Phase 2 — tokenise.py         (SentencePiece BPE, vocab=4000)
        │
        ├──▶ Phase 3 — train_baseline.py   (direct Si→Ta Transformer)
        │
        └──▶ Phase 4 — cascade_pivot.py    (Si→En via Helsinki + En→Ta via IndicTrans2)
                                │
                                ▼
Phase 5 — evaluate.py         (BLEU + chrF, comparison table)
```

---

**References used in this PoC:**
- Pramodya et al. (2024) — baseline architecture config (4L/4L/8H/d512)
- Zhang, Li & Liu (2022) — parameter freezing for cascade pivot training
- Gaikwad et al. (2024) — expected +2 to +5 BLEU from pivot on comparable pairs
- Feng et al. (2022) — LaBSE filtering threshold 0.85

In [1]:
import sys
from pathlib import Path

ROOT = Path('.').resolve()
SRC  = ROOT / 'src'
sys.path.insert(0, str(SRC))

print('Working directory:', ROOT)

Working directory: /home/pasindu/Downloads/research/proposal


## Phase 0 — Install Dependencies

Run once. Safe to skip if already installed.

In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
print('Dependencies installed.')

Dependencies installed.


## Phase 1 — Preprocessing

Loads raw Sinhala and Tamil files, applies:
- Unicode NFC normalisation (critical for legacy Sri Lankan government documents)
- Length filter: 5–100 tokens per side
- Deduplication on source side
- Train/test split with a fixed seed (50 sentences held out)

In [3]:
from preprocess import (
    load_parallel, clean_pairs, length_filter, deduplicate, split_test, write_parallel
)

raw_si = ROOT / 'data/raw/corpus.si'
raw_ta = ROOT / 'data/raw/corpus.ta'

print(f'Loading {raw_si} ...')
pairs = load_parallel(raw_si, raw_ta)
print(f'Loaded {len(pairs)} pairs')

pairs = clean_pairs(pairs)
pairs = [(si, ta) for si, ta in pairs if si and ta]
pairs = length_filter(pairs, min_tokens=2, max_tokens=60)
pairs = deduplicate(pairs)

train, test = split_test(pairs, test_size=20, seed=42)

write_parallel(train, ROOT / 'data/processed', 'train')
write_parallel(test,  ROOT / 'data/test_set',  'test')

print(f'\nFinal: {len(train)} train pairs, {len(test)} test pairs')

Loading /home/pasindu/Downloads/research/proposal/data/raw/corpus.si ...
Loaded 118 pairs
  Length filter: kept 118, dropped 0
  Deduplication: removed 0 exact source duplicates
  Split: 98 train, 20 test (seed=42)
  Wrote 98 pairs → /home/pasindu/Downloads/research/proposal/data/processed/train.si / /home/pasindu/Downloads/research/proposal/data/processed/train.ta
  Wrote 20 pairs → /home/pasindu/Downloads/research/proposal/data/test_set/test.si / /home/pasindu/Downloads/research/proposal/data/test_set/test.ta

Final: 98 train pairs, 20 test pairs


In [4]:
print('=== Sample train pairs ===')
for si, ta in train[:5]:
    print(f'  SI: {si}')
    print(f'  TA: {ta}')
    print()

=== Sample train pairs ===
  SI: ශ්‍රී ලංකාව දකුණු ආසියාවේ පිහිටි රටකි.
  TA: இலங்கை தெற்காசியாவில் அமைந்துள்ள நாடு.

  SI: කොළඹ ශ්‍රී ලංකාවේ වාණිජ අගනුවරයි.
  TA: கொழும்பு இலங்கையின் வணிக தலைநகரம்.

  SI: ශ්‍රී ලංකාවේ නිල භාෂා දෙකක් ඇත.
  TA: இலங்கையில் இரண்டு அலுவல் மொழிகள் உள்ளன.

  SI: සිංහල භාෂාව ශ්‍රී ලංකාවේ ප්‍රධාන භාෂාවයි.
  TA: சிங்களம் இலங்கையின் முக்கிய மொழியாகும்.

  SI: දෙමළ භාෂාව උතුරු සහ නැගෙනහිර පළාත්වල කතා කෙරේ.
  TA: தமிழ் மொழி வடக்கு மற்றும் கிழக்கு மாகாணங்களில் பேசப்படுகிறது.



## Phase 1b — LaBSE Quality Filtering

Filters sentence pairs where the LaBSE cosine similarity is below 0.85.

> **Note:** This step downloads the LaBSE model (~1.8GB) on first run.
> On CPU it takes ~2 minutes for 100 pairs.
> Set `SKIP_LABSE = True` to skip this step and use all pairs.

In the full research system this is applied to the mined trilingual corpus from Sri Lankan government websites.

In [5]:
SKIP_LABSE = False

from labse_filter import load_model, filter_by_labse

train_si = [si for si, _ in train]
train_ta = [ta for _, ta in train]

if not SKIP_LABSE:
    labse_model = load_model()
    filtered_si, filtered_ta, scores = filter_by_labse(
        train_si, train_ta, labse_model, threshold=0.85, batch_size=32
    )
    train_filtered = list(zip(filtered_si, filtered_ta))
    print(f'After LaBSE filter: {len(train_filtered)} pairs (from {len(train)})')
else:
    train_filtered = train
    scores = [1.0] * len(train)
    print(f'Skipped LaBSE — using all {len(train)} pairs')

write_parallel(train_filtered, ROOT / 'data/processed', 'train_filtered')

Loading LaBSE model (sentence-transformers/LaBSE) ...


/home/pasindu/Downloads/research/proposal/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding 98 Sinhala sentences ...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Encoding 98 Tamil sentences ...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

  LaBSE filter (threshold=0.85): kept 62, dropped 36 (36.7%)
  Score stats — mean: 0.9358, min: 0.8534, max: 0.9912
After LaBSE filter: 62 pairs (from 98)
  Wrote 62 pairs → /home/pasindu/Downloads/research/proposal/data/processed/train_filtered.si / /home/pasindu/Downloads/research/proposal/data/processed/train_filtered.ta


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if scores and not SKIP_LABSE:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(scores, bins=20, color='steelblue', edgecolor='white')
    ax.axvline(0.85, color='red', linestyle='--', label='Threshold (0.85)')
    ax.set_xlabel('LaBSE Cosine Similarity')
    ax.set_ylabel('Count')
    ax.set_title('LaBSE Similarity Distribution — Training Pairs')
    ax.legend()
    plt.tight_layout()
    plt.savefig(ROOT / 'results/labse_distribution.png', dpi=120)
    plt.show()
    print(f'Mean similarity: {np.mean(scores):.4f}')
else:
    print('LaBSE step skipped — no distribution to plot')

## Phase 2 — BPE Tokenisation

Trains SentencePiece BPE models separately for Sinhala and Tamil.

**Why separate vocabularies?**  
Sinhala (56 base characters) and Tamil (247 characters) use entirely different scripts.
A shared vocabulary would waste capacity and produce poor subword splits.

**Vocab size:**  
4,000 for PoC (data is small). Full research system: 8,000 (Tennage et al. 2017).

In [ ]:
from tokenise import train_spm, load_spm, encode_file

spm_dir = ROOT / 'data/processed/spm'
tok_dir = ROOT / 'data/processed/tokenised'
tok_dir.mkdir(parents=True, exist_ok=True)

si_train_path = ROOT / 'data/processed/train_filtered.si'
ta_train_path = ROOT / 'data/processed/train_filtered.ta'

print('Training Sinhala BPE model ...')
train_spm(si_train_path, spm_dir / 'si_spm', vocab_size=4000)

print('Training Tamil BPE model ...')
train_spm(ta_train_path, spm_dir / 'ta_spm', vocab_size=4000)

si_sp = load_spm(spm_dir / 'si_spm.model')
ta_sp = load_spm(spm_dir / 'ta_spm.model')

encode_file(si_sp, si_train_path, tok_dir / 'train_filtered.si')
encode_file(ta_sp, ta_train_path, tok_dir / 'train_filtered.ta')

print('Tokenisation complete.')

In [ ]:
from tokenise import encode_sentence

example_si = train_filtered[0][0] if train_filtered else 'ශ්‍රී ලංකාව දකුණු ආසියාවේ පිහිටි රටකි.'
example_ta = train_filtered[0][1] if train_filtered else 'இலங்கை தெற்காசியாவில் அமைந்துள்ள நாடு.'

si_pieces = encode_sentence(si_sp, example_si)
ta_pieces = encode_sentence(ta_sp, example_ta)

print('Original SI:', example_si)
print('BPE pieces: ', ' | '.join(si_pieces))
print()
print('Original TA:', example_ta)
print('BPE pieces: ', ' | '.join(ta_pieces))

## Phase 3 — Baseline Direct Transformer (Si→Ta)

Trains a minimal Transformer directly on the Sinhala-Tamil pairs.

**Architecture** (Pramodya et al. 2024):
- 4 encoder layers, 4 decoder layers, 8 attention heads, d_model=512
- Label smoothing 0.1, Adam with Noam LR schedule (warmup=400 for PoC)

**PoC caveat:** Only ~100 training pairs, 20 epochs. BLEU will be very low (3–8).
This demonstrates the *pipeline* and the data-scarcity ceiling, not final performance.

In [ ]:
import torch
from train_baseline import (
    build_vocab_from_spm_vocab, ParallelDataset, collate_fn,
    BaselineTransformer, train as train_epoch, greedy_translate
)
from torch.utils.data import DataLoader
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

si_vocab = build_vocab_from_spm_vocab(spm_dir / 'si_spm.model')
ta_vocab = build_vocab_from_spm_vocab(spm_dir / 'ta_spm.model')
print(f'SI vocab: {len(si_vocab)}, TA vocab: {len(ta_vocab)}')

In [ ]:
import math

dataset = ParallelDataset(
    tok_dir / 'train_filtered.si',
    tok_dir / 'train_filtered.ta',
    si_vocab, ta_vocab
)
loader = DataLoader(dataset, batch_size=8, shuffle=True,
                    collate_fn=lambda b: collate_fn(b, pad_id=3))
print(f'Dataset: {len(dataset)} pairs, {len(loader)} batches/epoch')

model = BaselineTransformer(
    src_vocab_size=len(si_vocab),
    tgt_vocab_size=len(ta_vocab),
    d_model=512, nhead=8,
    num_enc_layers=4, num_dec_layers=4,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

In [ ]:
EPOCHS     = 20
WARMUP     = 400
D_MODEL    = 512

criterion = nn.CrossEntropyLoss(ignore_index=3, label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)

def lr_lambda(step):
    step = max(step, 1)
    return D_MODEL ** -0.5 * min(step ** -0.5, step * WARMUP ** -1.5)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

history = []
for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(model, loader, optimizer, scheduler, criterion, device, epoch)
    history.append(loss)

print('\nBaseline training complete.')

baseline_dir = ROOT / 'models/baseline'
baseline_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), baseline_dir / 'model.pt')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, len(history)+1), history, color='steelblue')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Baseline Training Loss')
plt.tight_layout()
plt.savefig(ROOT / 'results/baseline_loss.png', dpi=120)
plt.show()

In [ ]:
test_si  = (ROOT / 'data/test_set/test.si').read_text(encoding='utf-8').splitlines()
test_ta  = (ROOT / 'data/test_set/test.ta').read_text(encoding='utf-8').splitlines()

unk_si = si_vocab.get('<unk>', 0)
bos_ta = ta_vocab.get('<s>', 1)
eos_ta = ta_vocab.get('</s>', 2)
id_to_ta = {v: k for k, v in ta_vocab.items()}

baseline_hyps = []
model.eval()
for si_sent in test_si:
    tokens = [si_vocab.get(tok, unk_si) for tok in si_sp.encode(si_sent, out_type=str)]
    output_ids = greedy_translate(model, tokens, bos_ta, eos_ta, device)
    pieces = [id_to_ta.get(i, '<unk>') for i in output_ids if i != eos_ta]
    baseline_hyps.append(ta_sp.decode(pieces))

(ROOT / 'models/baseline/hypothesis.ta').write_text('\n'.join(baseline_hyps), encoding='utf-8')
print(f'Generated {len(baseline_hyps)} baseline translations.')
print('\nFirst 3:')
for si, hyp, ref in zip(test_si[:3], baseline_hyps[:3], test_ta[:3]):
    print(f'  SRC: {si}')
    print(f'  REF: {ref}')
    print(f'  HYP: {hyp}')
    print()

## Phase 4 — Cascade Pivot Translation (Si→En→Ta)

**Step 1:** `Helsinki-NLP/opus-mt-si-en` — pretrained Marian MT model for Sinhala→English.  
**Step 2:** `ai4bharat/indictrans2-en-indic-1B` — IndicTrans2 for English→Tamil.

Both models are used **inference-only** in the PoC.  
In the full research system:
- Step 1 is fine-tuned on the government domain corpus
- Step 2 (IndicTrans2) is fine-tuned on English-Tamil government data
- Encoder parameters frozen after Stage 1 (Zhang et al. 2022)

> **Hardware note:** IndicTrans2 (1B params) requires ~6GB VRAM for inference.
> On CPU it is slow (~5-10 sec/sentence). Set `DEVICE = 'cuda'` if GPU is available.

In [ ]:
from cascade_pivot import load_si_en, translate_si_en, load_en_ta, translate_en_ta, freeze_encoder

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Cascade device: {DEVICE}')

print('\nStep 1: Loading Si→En model ...')
si_en_tok, si_en_model = load_si_en(DEVICE)

print('Translating Si→En ...')
intermediate_en = translate_si_en(test_si, si_en_tok, si_en_model, DEVICE, batch_size=8)

(ROOT / 'models/cascade').mkdir(parents=True, exist_ok=True)
(ROOT / 'models/cascade/intermediate_en.txt').write_text('\n'.join(intermediate_en), encoding='utf-8')

print('\nSample Si→En:')
for si, en in zip(test_si[:3], intermediate_en[:3]):
    print(f'  SI: {si}')
    print(f'  EN: {en}')
    print()

In [ ]:
print('Demonstrating parameter freezing (Zhang et al. 2022):')
trainable_before = sum(p.numel() for p in si_en_model.parameters() if p.requires_grad)
print(f'  Trainable params before freeze: {trainable_before:,}')

freeze_encoder(si_en_model)

trainable_after = sum(p.numel() for p in si_en_model.parameters() if p.requires_grad)
print(f'  Trainable params after  freeze: {trainable_after:,}')
print(f'  Frozen: {trainable_before - trainable_after:,} parameters')

In [ ]:
del si_en_model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

print('Step 2: Loading En→Ta model (IndicTrans2) ...')
en_ta_tok, en_ta_model = load_en_ta(DEVICE)

print('Translating En→Ta ...')
cascade_ta = translate_en_ta(intermediate_en, en_ta_tok, en_ta_model, DEVICE, batch_size=4)

(ROOT / 'models/cascade/final_ta.txt').write_text('\n'.join(cascade_ta), encoding='utf-8')

print('\nSample Si→En→Ta:')
for si, en, ta, ref in zip(test_si[:3], intermediate_en[:3], cascade_ta[:3], test_ta[:3]):
    print(f'  SI  : {si}')
    print(f'  EN  : {en}')
    print(f'  TA  : {ta}')
    print(f'  REF : {ref}')
    print()

## Phase 5 — Evaluation: BLEU + chrF

Both systems evaluated on the same fixed held-out test set.

**Expected PoC results:**
| System | BLEU | chrF |
|--------|------|------|
| Baseline (direct, tiny model) | 3–8 | 15–25 |
| Cascade (pretrained Si→En→Ta) | 12–22 | 30–45 |

The cascade advantage demonstrates the core research insight: routing through
a high-resource pivot language with pretrained models overcomes the 25k-pair ceiling.

In [ ]:
from evaluate import score_system, make_comparison_md
import json

results_dir = ROOT / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

references    = (ROOT / 'data/test_set/test.ta').read_text(encoding='utf-8').splitlines()
baseline_hyps = (ROOT / 'models/baseline/hypothesis.ta').read_text(encoding='utf-8').splitlines()
cascade_hyps  = (ROOT / 'models/cascade/final_ta.txt').read_text(encoding='utf-8').splitlines()
cascade_en    = (ROOT / 'models/cascade/intermediate_en.txt').read_text(encoding='utf-8').splitlines()

print('=== Automatic Evaluation ===')
baseline_scores = score_system('Baseline', baseline_hyps, references)
cascade_scores  = score_system('Cascade',  cascade_hyps,  references)

(results_dir / 'baseline_scores.json').write_text(json.dumps(baseline_scores, indent=2))
(results_dir / 'cascade_scores.json').write_text(json.dumps(cascade_scores, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
systems = ['Baseline\n(direct Si→Ta)', 'Cascade\n(Si→En→Ta)']
colors  = ['#d9534f', '#5cb85c']

bleu_scores = [baseline_scores['bleu'], cascade_scores['bleu']]
chrf_scores = [baseline_scores['chrf'], cascade_scores['chrf']]

axes[0].bar(systems, bleu_scores, color=colors, edgecolor='white', width=0.5)
axes[0].set_ylabel('BLEU')
axes[0].set_title('BLEU Score Comparison')
axes[0].axhline(21.26, color='gray', linestyle='--', linewidth=0.8, label='SOTA baseline (21.26)')
axes[0].axhline(30,    color='navy', linestyle=':',  linewidth=0.8, label='Practical threshold (30)')
axes[0].legend(fontsize=8)
for i, v in enumerate(bleu_scores):
    axes[0].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=11, fontweight='bold')

axes[1].bar(systems, chrf_scores, color=colors, edgecolor='white', width=0.5)
axes[1].set_ylabel('chrF')
axes[1].set_title('chrF Score Comparison')
for i, v in enumerate(chrf_scores):
    axes[1].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Sinhala-Tamil MT: Baseline vs Cascade Pivot', fontweight='bold')
plt.tight_layout()
plt.savefig(results_dir / 'score_comparison.png', dpi=150)
plt.show()

delta_bleu = cascade_scores['bleu'] - baseline_scores['bleu']
delta_chrf = cascade_scores['chrf'] - baseline_scores['chrf']
print(f'\nDelta BLEU : {delta_bleu:+.2f}')
print(f'Delta chrF : {delta_chrf:+.2f}')

In [ ]:
report = make_comparison_md(
    baseline_scores, cascade_scores,
    test_si, references, baseline_hyps, cascade_hyps, cascade_en,
    n_samples=10
)
(results_dir / 'comparison.md').write_text(report, encoding='utf-8')
print('Report written to results/comparison.md')
print()
print(report[:2000])

## Summary

This PoC has demonstrated the end-to-end cascade pivot pipeline:

1. **Data preprocessing** — Unicode normalisation, length filtering, deduplication, train/test split
2. **LaBSE filtering** — quality-based parallel sentence selection (threshold 0.85)
3. **BPE tokenisation** — SentencePiece with separate Sinhala/Tamil vocabularies
4. **Baseline model** — tiny direct Transformer, demonstrates the data-scarcity ceiling
5. **Cascade pivot** — Helsinki-NLP (Si→En) + IndicTrans2 (En→Ta), shows the advantage of routing through a high-resource pivot
6. **Evaluation** — BLEU + chrF via sacreBLEU, comparison chart, full report

---

### Next Steps for Full Research

| Step | Action |
|------|---------|
| Replace sample data | Download real Sinhala-Tamil corpus (~25k pairs) + FLORES-200 + Samanantar |
| Fine-tune Si→En | Train `Helsinki-NLP/opus-mt-si-en` on government domain corpus |
| Fine-tune En→Ta | Fine-tune IndicTrans2 on Tamil-English government data |
| Apply parameter freezing | Call `freeze_encoder()` after Stage 1 training |
| Scale evaluation | 500-sentence fixed test set + human evaluation (200 sentences) |
| Publish | Release test set + model checkpoints for reproducibility |